### Import libraries and setting a constant

In [2]:
import requests
import os
import re
import pandas as pd
import numpy as np
import logging
import time
import random
from dateutil import parser
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

In [3]:
BASE_URL = "https://liquipedia.net/mobilelegends/"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def get_soup(url):
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")

def absolute(href, base=BASE_URL):
    return urljoin(base, href)

In [4]:
def get_url(content) -> set:
    try:
        if content is None:
            return set()
        results = set()
        item = content.select("a")
        for i in item:
            href = i.get("href")
            if href:
                if href.startswith("/mobilelegends/") or href.startswith("/"):
                    url = absolute(href)
                    results.add(url)
        return list(results)
    
    except Exception as e:
        print(f"Error occur when running get_url method: {e}")

### Getting stage per tournament URL

Scraping stage url per tournament through tournament portal page, then filter selected url using some keyword filter.

In [5]:
current_url = "https://liquipedia.net/mobilelegends/portal:tournaments"
soup = get_soup(current_url)
content = soup.select_one(".nav-tabs")
urls = get_url(content)
tier = []
for url in urls:
    parse = urlparse(url).path
    if any(t in parse.replace("/", " ").lower() for t in ["s-tier", "a-tier", "b-tier", "qualifier_tournament"]):
        tier.append([parse.split("/")[-1].replace("_", " "), url])
print(tier)

[['A-Tier Tournaments', 'https://liquipedia.net/mobilelegends/A-Tier_Tournaments'], ['S-Tier Tournaments', 'https://liquipedia.net/mobilelegends/S-Tier_Tournaments'], ['Qualifier Tournaments', 'https://liquipedia.net/mobilelegends/Qualifier_Tournaments'], ['B-Tier Tournaments', 'https://liquipedia.net/mobilelegends/B-Tier_Tournaments']]


In [6]:
current_url = "https://liquipedia.net/mobilelegends/A-Tier_Tournaments"
soup = get_soup(current_url)
tournament = []
urls = []
item = soup.select("tr.table2__row--body")
for i in item:
    content = get_url(i)
    urls.extend(content)
for url in urls:
    parse = urlparse(url).path
    if any(t in parse.replace("/", " ").lower() for t in ["indonesia season_16", "indonesia season_15"]):
        tournament.append([" ".join(parse.split("/")[2:]).replace("_", " "), url])
print(tournament)

[['MPL Indonesia Season 16', 'https://liquipedia.net/mobilelegends/MPL/Indonesia/Season_16'], ['MPL Indonesia Season 15', 'https://liquipedia.net/mobilelegends/MPL/Indonesia/Season_15']]


In [9]:
current_url = "https://liquipedia.net/mobilelegends/MPL/Indonesia/Season_16"
soup = get_soup(current_url)
content = soup.select_one(".nav-tabs")
urls = get_url(content)
stage = []
for url in urls:
    parse = urlparse(url).path
    if any(t in parse.replace("/", " ").lower() for t in ["regular_season", "group_stage", "playoffs", "knockout_stage"]):
        stage.append([parse.split("/")[-1].replace("_", " "), url])
print(stage)
if not any(link.startswith(current_url) for [_, link] in stage):   
    print(f"there is no url. Adding {current_url} as representation.")
        

[['Playoffs', 'https://liquipedia.net/mobilelegends/MPL/Indonesia/Season_16/Playoffs'], ['Regular Season', 'https://liquipedia.net/mobilelegends/MPL/Indonesia/Season_16/Regular_Season']]


### Getting match detail

Scraping match detail through stage URL that we scrape earlier. Get information such team, opponent, winner, picks, bans, duration, etc. Since each stage has its own format, divide method based on it.

#### Elimination Format

Single elimination has bracket level to determine match position. So, Need to determine match position based it's bracket level. I use bottom up method to get bracket level through selector path. Then, start to scraping match detail.

In [10]:
def get_bracket(soup, selector=None, depth=0, max_depth=15):
    if soup is None or depth > max_depth:
        return None, selector
    
    parent = soup.find_parent("div")

    if parent is None:
        return None, selector
    
    parent_classes = parent.get("class") or []
    if not parent_classes:
        return None, selector
    
    sibling = parent.find_previous_sibling("div")
    
    if selector is None:
        selector = f".{parent_classes[0]}"
    else:
        selector = f".{parent_classes[0]} > {selector}"
    
    if sibling:
        sibling_classes = sibling.get("class") or []
        if any(c.endswith("header") for c in sibling_classes):
            depth_to_level ={
                1: "Grand Final",
                3: "Final",
                5: "Semi-Final",
                7: "Quarter-Final",
                9: "Round 1",
            }
            level = depth_to_level.get(depth)

            if "upper" in sibling.get_text().lower() and depth > 1:
                bracket = f"Upper {level}"
            elif "lower" in sibling.get_text().lower() and depth > 1:
                bracket = f"Lower {level}"
            else:
                bracket = f"{level}"
            
            logging.info(f"Get_bracket completed. Found [{selector}] for {bracket} bracket.")
            return True, bracket
        
        sibling = sibling.find_previous_sibling("div")
        if sibling:
            sibling_classes = sibling.get("class") or []
            if any(c.endswith("header") for c in sibling_classes):
                depth_to_level ={
                    1: "Grand Final",
                    3: "Final",
                    5: "Semi-Final",
                    7: "Quarter-Final",
                    9: "Round 1",
                }
                level = depth_to_level.get(depth)

                if "upper" in sibling.get_text().lower() and depth > 1:
                    bracket = f"Upper {level}"
                elif "lower" in sibling.get_text().lower() and depth > 1:
                    bracket = f"Lower {level}"
                else:
                    bracket = f"{level}"
                
                logging.info(f"Get_bracket completed. Found [{selector}] for {bracket} bracket.")
                return True, bracket
    depth += 1
    return False, parent, selector, depth



##### Match 

Match bracket are important features as variable to determine team ELO rating. 

In [11]:
soup = get_soup("https://liquipedia.net/mobilelegends/MPL/Indonesia/Season_16/Playoffs")
matches = soup.select(".brkts-match")
bracket = get_bracket(soup=matches[0])
while bracket[0] == False:
    bracket = get_bracket(soup=bracket[1], selector=bracket[2], depth=bracket[3])
    bracket_name = bracket[1]
bracket_name

'Upper Quarter-Final'

##### Match details

In [12]:
soup = matches[0]
results = []
popups = soup.select(".brkts-match-info-popup")
print(popups)
popup = popups[0]
timestamp = popup.select_one(".timer-object-datetime-only")
date = timestamp.get_text()
date = parser.parse(date)
date = date.date()

teams = popup.select(".name.hidden-xs")
home_team = teams[0].get_text()
away_team = teams[1].get_text()

games = popup.select(".brkts-popup-body-grid-row")
if len(games) == 0:
    games = popup.select(".brkts-popup-body-game")

# ambil teks score atau result
score_el = popup.select_one(".match-info-header-scoreholder-upper")
score_text = score_el.get_text().strip() if score_el else ""

# cek apakah ini auto-win
is_autowin = any(k in score_text for k in ["Win", "Winner", "FF", "DQ"])
is_empty_game = (len(games) == 0) or (len(games) == 1 and not games[0].get_text.strip())

if is_autowin or is_empty_game:
    print("FF/Autowin detected.")

bans_item = popup.select(".brkts-popup-mapveto__ban-round")
all_bans = []
for b in bans_item:
    num = b.get_text().replace("\xa0", " ").strip()[-1]
    idx = int(num) - 1
    while len(all_bans) <= idx:
        all_bans.append([])
    
    ban_heroes = [a.get("title") for a in b.select("a")]
    all_bans[idx] = ban_heroes

home_bans, away_bans = [], []
for i, bans in enumerate(all_bans):
    if bans == []:
        home_bans.append([])
        away_bans.append([])
    else:
        mid = len(bans) // 2
        home, away = bans[:mid], bans[mid:]
        home_bans.append(home)
        away_bans.append(away)
    
for i, game in enumerate(games, start=1):
    time = game.select_one(".brkts-popup-body-grid-row-detail")
    if not time:
        time = game.select_one(".brkts-popup-spaced")
    duration = time.get_text()
    map = game.select_one(".brkts-popup-comment")
    map_name = "default"
    if map:
        map_name = map.get_text()
        if map_name == "":
            map_name = "unknown"
    if ":" not in duration:
        duration = "unknown"
        map_name = "unknown"

    print(f"{duration} and {map_name}")
    icon = game.select_one(".generic-label")
    classes = icon.get("data-label-type", []) if icon else []
    if icon is None:
        icon = game.select_one(".brkts-result-label")
        classes = icon.get("class", []) if icon else []
    # print(icon)
    # print(classes)

    pick_heroes = [a.get("title") for a in game.select("a")]
    home_picks, away_picks = pick_heroes[:5], pick_heroes[5:]
    
    results.append({
        "date": date,
        "game_num": i,
        "home_team": home_team,
        "away_team": away_team,
        "home_picks": home_picks,
        "away_picks": away_picks,
        "home_bans": home_bans[i-1] if i-1 < len(home_bans) else [],
        "away_bans": away_bans[i-1] if i-1 < len(away_bans) else [],
        "duration": duration,
        "map_name": map_name,
        "home_status": "win" if "win" in classes else "loss",
        "away_status": "loss" if "win" in classes else "win",
        # "tier": tier,
        # "tournament": tournament,
        # "stage": stage
    })
df = pd.DataFrame(results)
df

[<div class="brkts-popup brkts-popup-container brkts-match-info-popup" data-analytics-name="Match popup" style="width:420px"><span class="match-info-countdown"><span class="timer-object timer-object-datetime-only" data-finished="finished" data-format="full" data-timestamp="1761717600">October 29, 2025 - 13:00 <abbr data-tz="+07:00" title="Indochina Time (UTC+7)">ICT</abbr></span></span><div class="match-info-header"><div class="match-info-header-opponent match-info-header-opponent-left match-info-header-winner"><div class="block-team flipped"><span class="team-template-image-icon"><a href="/mobilelegends/Alter_Ego" title="Alter Ego"><img alt="" decoding="async" height="50" src="/commons/images/thumb/e/ea/Alter_Ego_2022_allmode.png/51px-Alter_Ego_2022_allmode.png" srcset="/commons/images/thumb/e/ea/Alter_Ego_2022_allmode.png/76px-Alter_Ego_2022_allmode.png 1.5x, /commons/images/thumb/e/ea/Alter_Ego_2022_allmode.png/102px-Alter_Ego_2022_allmode.png 2x" style="vertical-align: middle" widt

C:\Users\ACER\AppData\Roaming\Python\Python311\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname ICT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


,date,game_num,home_team,away_team,home_picks,away_picks,home_bans,away_bans,duration,map_name,home_status,away_status
0,2025-10-29,1,Alter Ego,Natus Vincere,"[Yu Zhong, Yi Sun-shin, Yve, Cici, Hylos]","[Lapu-Lapu, Fredrinn, Kimmy, Harith, Grock]","[Esmeralda, Kalea, Gatotkaca, Lancelot, Hayabusa]","[Baxia, Helcurt, Zhuxin, Valentina, Lunox]",15:34,Dangerous Grass,loss,win
1,2025-10-29,2,Alter Ego,Natus Vincere,"[Yu Zhong, Yi Sun-shin, Yve, Granger, Grock]","[Lapu-Lapu, Guinevere, Kimmy, Obsidia, Gatotkaca]","[Esmeralda, Kalea, Harith, Lancelot, Hayabusa]","[Baxia, Zhuxin, Cici, Lunox, Pharsa]",16:05,Broken Walls,win,loss
2,2025-10-29,3,Alter Ego,Natus Vincere,"[Arlott, Yi Sun-shin, Valentina, Uranus, Gatot...","[Alice, Fanny, Kimmy, Cici, Khaleed]","[Esmeralda, Harith, Grock, Lancelot, Yu Zhong]","[Baxia, Zhuxin, Helcurt, X.Borg, Ruby]",19:34,Dangerous Grass,win,loss
3,2025-10-29,4,Alter Ego,Natus Vincere,"[Uranus, Hayabusa, Yve, Claude, Kalea]","[Yu Zhong, Yi Sun-shin, Zhuxin, Bruno, Hylos]","[Esmeralda, Baxia, Cici, Ruby, Granger]","[Fanny, Grock, Gatotkaca, Lunox, Harith]",11:00,Expanding Rivers,win,loss


#### Round Robin Format

Since round robin doesn't use bracket, so let's scraping match details from match popup which total popup count as total match and it sorted by itself.

In [ ]:
current_url = "https://liquipedia.net/mobilelegends/MPL/Singapore/Season_9/Regular_Season"
soup = get_soup(current_url)
matches = soup.select(".brkts-match-has-details")
match = matches[4]
results = []
popups = match.select(".brkts-match-info-popup")
# print(popups)
popup = popups[0]
timestamp = popup.select_one(".timer-object-datetime-only")
date = timestamp.get_text()
date = parser.parse(date)
date = date.date()

teams = popup.select(".name.visible-xs a")
home_href = teams[0].get("href")
if "index" in home_href:
    home_name = teams[0].get("title")

else:
    home_url = urljoin(base=BASE_URL, url=home_href)
    home_soup = get_soup(home_url)
    name_box = home_soup.select_one("h1.firstHeading")
    home_name = name_box.get_text()

home_name = re.sub(r"\(.*?\)", "", home_name).strip()
home_alias = teams[0].get_text()

away_href = teams[1].get("href")
if "index" in away_href:
    away_name = teams[1].get("title")

else:
    away_url = urljoin(base=BASE_URL, url=away_href)
    away_soup = get_soup(away_url)
    name_box = away_soup.select_one("h1.firstHeading")
    away_name = name_box.get_text()

away_name = re.sub(r"\(.*?\)", "", away_name).strip()
away_alias = teams[1].get_text()

games = popup.select(".brkts-popup-body-grid-row")
if len(games) == 0:
    games = popup.select(".brkts-popup-body-game")
    if len(games) == 0:
        logging.info(f"This match does not have a statistic record.")
        score_box = popup.select(".match-info-header-scoreholder-upper")
        results.append({
            "date": date,
            "game_num": np.nan,
            "home_team": home_name,
            "home_alias": home_alias,
            "away_team": away_name,
            "away_alias": away_alias,
            "home_picks": [],
            "away_picks": [],
            "home_bans": [],
            "away_bans": [],
            "duration": np.nan,
            "map": np.nan,
            "home_status": "win" if "1" in score_box[0].get_text() else "loss",
            "away_status": "loss" if "1" in score_box[0].get_text() else "win",
            # "tier": tier,
            # "tournament": tournament,
            # "stage": stage,
            # "bracket": bracket
        })

# ambil teks score atau result
score_el = popup.select_one(".match-info-header-scoreholder-upper")
score_text = score_el.get_text().strip() if score_el else ""

# cek apakah ini auto-win
is_autowin = any(k in score_text for k in ["Win", "Winner", "FF", "DQ"])
is_empty_game = (len(games) == 0) or (len(games) == 1 and not games[0].get_text.strip())

if is_autowin or is_empty_game:
    print("FF/Autowin detected.")

bans_item = popup.select(".brkts-popup-mapveto__ban-round")
all_bans = []
for b in bans_item:
    num = b.get_text().replace("\xa0", " ").strip()[-1]
    idx = int(num) - 1
    while len(all_bans) <= idx:
        all_bans.append([])
    
    ban_heroes = [a.get("title") for a in b.select("a")]
    all_bans[idx] = ban_heroes

home_bans, away_bans = [], []
for i, bans in enumerate(all_bans):
    if bans == []:
        home_bans.append([])
        away_bans.append([])
    else:
        mid = len(bans) // 2
        home, away = bans[:mid], bans[mid:]
        home_bans.append(home)
        away_bans.append(away)
    
for i, game in enumerate(games, start=1):
    time = game.select_one(".brkts-popup-body-grid-row-detail")
    if not time:
        time = game.select_one(".brkts-popup-spaced")
    duration = time.get_text()
    map = game.select_one(".brkts-popup-comment")
    map_name = "default"
    if map:
        map_name = map.get_text()
        if map_name == "":
            map_name = "unknown"
    if ":" not in duration:
        duration = "unknown"
        map_name = "unknown"

    print(f"{duration} and {map_name}")
    icon = game.select_one(".generic-label")
    classes = icon.get("data-label-type", []) if icon else []
    if icon is None:
        icon = game.select_one(".brkts-result-label")
        classes = icon.get("class", []) if icon else []
    # print(icon)
    # print(classes)

    pick_heroes = [a.get("title") for a in game.select("a")]
    home_picks, away_picks = pick_heroes[:5], pick_heroes[5:]
    
    results.append({
        "date": date,
        "game_num": i,
        "home_team": home_name,
        "home_alias": home_alias,
        "away_team": away_name,
        "away_alias": away_alias,
        "home_picks": home_picks,
        "away_picks": away_picks,
        "home_bans": home_bans[i-1] if i-1 < len(home_bans) else [],
        "away_bans": away_bans[i-1] if i-1 < len(away_bans) else [],
        "duration": duration,
        "map_name": map_name,
        "home_status": "win" if "win" in classes else "loss",
        "away_status": "loss" if "win" in classes else "win",
        # "tier": tier,
        # "tournament": tournament,
        # "stage": stage
    })
df = pd.DataFrame(results)
df

C:\Users\ACER\AppData\Roaming\Python\Python311\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname SGT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


,date,game_num,home_team,home_alias,away_team,away_alias,home_picks,away_picks,home_bans,away_bans,duration,map,home_status,away_status
0,2025-05-11,NaN,Absolute Zero,AZ,Sovereign SG,SVG,[],[],[],[],NaN,NaN,win,loss


#### Notes

Since tournament format will be change in the future, the fundamental is the format using bracket feature or not. if they use it, choose the elimination format method. the other hand, choose the round robin format method. for example, the tournament using GSL format which is using both. So, just choose one of them to use to prevent duplicated result.

### Getting list of team participant

there are two difference method to get team participant list. first, scraping through team portal page on main page. Second, scraping through tournament page that we scrape earlier. I prefer the second because team portal page contain all the team exist in the scene. But, all we need just the team who participate in certain tournament.

In [ ]:
all_team = []
current_url = "https://liquipedia.net/mobilelegends/Games_of_the_Future/2025"
soup = get_soup(current_url)
region = soup.select_one(".fo-nttax-infobox")
cards = soup.select(".teamcard.toggle-area")
# # result = set()
if not cards:
    cards = soup.select(".team-participant-card__opponent-compact")
for card in cards:
    content = card.select("center a[href]")
    if not content:
        content = card.select("span.name a[href]")

    href = content[-1].get("href")
    team_url = urljoin(base=BASE_URL, url=href)
    # print(team_url)
    if "index" in href:
        team_name = content[-1].get_text()
        team_name = re.sub(r"\(.*?\)", "", team_name).strip()
        flag = region.select_one("span.flag a[title]")
        if flag:
            team_region = flag.get("title")
            print(f"{team_name}, {team_region}")
            continue

    team_soup = get_soup(team_url)
    team = team_soup.select_one("h1.firstHeading")
    team_name = team.get_text()
    if any(team_name in [t,_] for [t,_] in all_team):
        print(f"{team_name} already exists.")
        continue
    infobox = team_soup.select_one(".fo-nttax-infobox")
    if infobox:
        info = infobox.select(".infobox-description")
        text = []
        for i in info:
            text_info = i.get_text()
            text.append(text_info)

        flag = infobox.select("span.flag a[title]")
        # print(len(flag))
        if any("region" or "location" in t.lower() for t in text):
            team_region = flag[-1].get("title")
            if "asia" in team_region.lower() or "europe" in team_region.lower():
                team_region = flag[0].get("title")
            print(f"{team_name}, {team_region}")

        else:
            team_region = flag[0].get("title")
            print(f"{team_name}, {team_region}")
# result

{('Aurora Gaming', 'United Arab Emirates'),
 ('Aurora PH', 'United Arab Emirates'),
 ('DianFengYaoGuai', 'United Arab Emirates'),
 ('E7', 'United Arab Emirates'),
 ('HomeBois', 'United Arab Emirates'),
 ('INFLUENCE RAGE', 'United Arab Emirates'),
 ('Mythic SEAL', 'United Arab Emirates'),
 ('ONIC', 'United Arab Emirates'),
 ('RRQ Hoshi', 'United Arab Emirates'),
 ('Team Falcons', 'United Arab Emirates'),
 ('Team Max', 'United Arab Emirates'),
 ('Team Spirit', 'United Arab Emirates'),
 ('Team Vamos', 'United Arab Emirates'),
 ('The MongolZ', 'United Arab Emirates'),
 ('Verso Time', 'United Arab Emirates'),
 ('ZETA DIVISION', 'United Arab Emirates')}

### Getting list of MLBB Heroes

Scraping MLBB heroes through MLBB heroes portal page. Get information such heroes name and they role.

In [14]:
soup = get_soup("https://liquipedia.net/mobilelegends/Portal:Heroes")
contents = soup.select(".content3")
for content in contents:
    roles = content.select(".white-text")
    for role in roles:
        role_name = role.get_text().split()[0]
        for hero in role.select('.zoom-container'):
            heroes_name = hero.get_text()
            print(f"{heroes_name} as {role_name}")

Akai as Tank
Alice as Tank
Atlas as Tank
Barats as Tank
Baxia as Tank
Belerick as Tank
Carmilla as Tank
Chip as Tank
Edith as Tank
Esmeralda as Tank
Franco as Tank
Fredrinn as Tank
Gatotkaca as Tank
Gloo as Tank
Grock as Tank
Hilda as Tank
Hylos as Tank
Johnson as Tank
Khufra as Tank
Lolita as Tank
Masha as Tank
Minotaur as Tank
Terizla as Tank
Tigreal as Tank
Uranus as Tank
Aldous as Fighter
Alpha as Fighter
Alucard as Fighter
Argus as Fighter
Arlott as Fighter
Aulus as Fighter
Badang as Fighter
Balmond as Fighter
Bane as Fighter
Barats as Fighter
Benedetta as Fighter
Chou as Fighter
Cici as Fighter
Dyrroth as Fighter
Fredrinn as Fighter
Freya as Fighter
Gatotkaca as Fighter
Grock as Fighter
Guinevere as Fighter
Hilda as Fighter
Jawhead as Fighter
Julian as Fighter
Kaja as Fighter
Kalea as Fighter
Khaleed as Fighter
Lapu-Lapu as Fighter
Leomord as Fighter
Lukas as Fighter
Martis as Fighter
Masha as Fighter
Minsitthar as Fighter
Paquito as Fighter
Phoveus as Fighter
Roger as Fighter
Ru

### Getting list of patches

Scraping MLBB heroes through MLBB heroes portal page. Get information such patches note and release date.

In [32]:
current_url = "https://liquipedia.net/mobilelegends/portal:patches"
result = []
soup = get_soup(current_url)
tables = soup.select(".wikitable.collapsible")
for table in tables[:2]:
    rows = table.select("td a[href]")
    for item in rows:
        patch = item.get("title")
        if not patch or not "Patch" in patch:
            continue
        
        date = item.find_parent("td").find_next_sibling("td")
        date = date.get_text()
        date = parser.parse(date)
        date = date.date().strftime("%d-%m-%Y")
        result.append({
            "patch": patch,
            "release_date": date
        })

result

[{'patch': 'Patch 2.1.61', 'release_date': '11-03-2026'},
 {'patch': 'Patch 2.1.47', 'release_date': '28-01-2026'},
 {'patch': 'Patch 2.1.40a', 'release_date': '31-12-2025'},
 {'patch': 'Patch 2.1.40', 'release_date': '18-12-2025'},
 {'patch': 'Patch 2.1.30a', 'release_date': '27-11-2025'},
 {'patch': 'Patch 2.1.30', 'release_date': '04-11-2025'},
 {'patch': 'Patch 2.1.18b', 'release_date': '24-10-2025'},
 {'patch': 'Patch 2.1.18a', 'release_date': '29-09-2025'},
 {'patch': 'Patch 2.1.18', 'release_date': '17-09-2025'},
 {'patch': 'Patch 1.9.99a', 'release_date': '26-08-2025'},
 {'patch': 'Patch 1.9.99', 'release_date': '05-08-2025'},
 {'patch': 'Patch 1.9.90', 'release_date': '18-06-2025'},
 {'patch': 'Patch 1.9.64C', 'release_date': '14-05-2025'},
 {'patch': 'Patch 1.9.64B', 'release_date': '22-04-2025'},
 {'patch': 'Patch 1.9.64', 'release_date': '18-03-2025'},
 {'patch': 'Patch 1.9.42E', 'release_date': '13-02-2025'},
 {'patch': 'Patch 1.9.42D', 'release_date': '07-02-2025'},
 {'pa